<a href="https://colab.research.google.com/github/katherinexxxl/ADA-Assignment-Group-36/blob/main/Advanced_Data_Analytics_Group36_Q3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ConfusionMatrixDisplay as CM
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [39]:
# Upload File
from google.colab import files
uploaded = files.upload()

Saving early_2012_2013_loan_sample_with_outcome.csv to early_2012_2013_loan_sample_with_outcome (2).csv


In [40]:
# Load Dataset
df = pd.read_csv("early_2012_2013_loan_sample_with_outcome.csv")
print(df.describe())

                 id     member_id     loan_amnt   funded_amnt  \
count  5.000000e+04  5.000000e+04  50000.000000  50000.000000   
mean   1.918444e+06  2.283786e+06  13901.202000  13896.232000   
std    6.389890e+05  8.024902e+05   8086.390575   8081.255517   
min    5.852400e+04  1.495120e+05   1000.000000   1000.000000   
25%    1.443048e+06  1.695278e+06   8000.000000   8000.000000   
50%    1.587758e+06  1.857296e+06  12000.000000  12000.000000   
75%    2.311939e+06  2.744578e+06  19200.000000  19200.000000   
max    3.304574e+06  4.076727e+06  35000.000000  35000.000000   

       funded_amnt_inv          term      int_rate   installment  \
count     50000.000000  50000.000000  50000.000000  50000.000000   
mean      13877.691336     40.492320     13.997130    436.954180   
std        8072.373068      9.361437      4.280414    245.782294   
min         950.000000     36.000000      6.000000     25.810000   
25%        7950.000000     36.000000     11.140000    255.660000   
50%   

In [41]:
# Data Cleaning
# Check missing values
missing_counts = df.isna().sum()
print("\nMissing values per column:")
print(missing_counts)
# Note: 3 columns (tot_coll_amt, tot_cur_bal, total_credit_rv) share 14,618 missing values. Other blanks are acceptable given feature definitions.
# Check if all 3 columns' missing values occur in the same rows
co_missing = df[["tot_coll_amt", "tot_cur_bal", "total_credit_rv"]].isna().all(axis=1).sum()
print(f"\nRows where tot_coll_amt, tot_cur_bal AND total_credit_rv are all NA: {co_missing}")
# All missing values in the 3 features are contained in the same records.

# Data encoding & coversions
# Remove "%" from revol_util and convert to numeric
df["revol_util"] = pd.to_numeric(
    df["revol_util"].astype(str).str.replace("%", "", regex=False),
    errors="coerce")

# Convert loan_is_bad from "TRUE"/"FALSE" to 1/0 integer
df["loan_is_bad"] = np.where(df["loan_is_bad"] == 'TRUE', 1, 0)

# Create ordinal grade variable: A=1, B=2, ..., G=7
grade_order = ["A", "B", "C", "D", "E", "F", "G"]
df["grade_ord"] = df["grade"].map({g: i + 1 for i, g in enumerate(grade_order)})

# Convert all remaining object/string columns to 'category' dtype
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype("category")


Missing values per column:
id                                 0
member_id                          0
loan_amnt                          0
funded_amnt                        0
funded_amnt_inv                    0
term                               0
int_rate                           0
installment                        0
grade                              0
sub_grade                          0
emp_title                       2832
emp_length                      1802
home_ownership                     0
annual_inc                         0
verification_status                0
issue_d                            0
loan_status                        0
pymnt_plan                         0
desc                           18996
purpose                            0
title                              2
zip_code                           0
addr_state                         0
dti                                0
delinq_2yrs                        0
earliest_cr_line                   0
inq_last_6

In [42]:
# Selection
# Selecting features that are relevant for predicting loan default, only including information available at the loan decision point. Removing variables such as     "out_prncp", "out_prncp_inv", "total_pymnt", "total_pymnt_inv", "total_rec_prncp", "total_rec_int", "total_rec_late_fee", "recoveries", "collection_recovery_fee", "last_pymnt_amnt"
# Only information available at the loan decision point is included.
select = [
    "loan_amnt",
    "int_rate",
    "annual_inc",
    "dti",
    "delinq_2yrs",
    "inq_last_6mths",
    "revol_util",
    "total_acc",
    "grade_ord",
    "loan_is_bad"
]

df_select = df[select].copy()

In [48]:
# Checking Data Post Selection
missing_counts_select = df_select.isna().sum()
print(missing_counts_select)

# Dropping NAs
df_select = df_select.dropna()

loan_amnt         0
int_rate          0
annual_inc        0
dti               0
delinq_2yrs       0
inq_last_6mths    0
revol_util        0
total_acc         0
grade_ord         0
loan_is_bad       0
dtype: int64


In [49]:
# Data Partitioning
X = df_select.drop("loan_is_bad", axis = 1) # independent features
y = df_select["loan_is_bad"] # target

# Split 1: Separate Test set (15%) from the rest (85%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)

# Split 2: Separate Train (70% of total) and Validation (15% of total)
# 0.15 / 0.85 approx 0.176
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1764, random_state=42, stratify=y_temp)

# Scale the data (fit only on training data to prevent data leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [50]:
# Data Balancing
# Check for data imbalance in target
print(y.value_counts())
print("\nProportions:")
print(y.value_counts(normalize=True))
value_counts = y.value_counts()
imbalance_ratio = value_counts.min() / value_counts.max()
print(f"\nImbalance ratio in target: {imbalance_ratio:.4f}")

# Data balancing for training data using SMOTE
random_seed = 123
target_ratio = 0.67 # minority/majority
smote = SMOTE(sampling_strategy = target_ratio, random_state = random_seed)

X_smote, y_smote = smote.fit_resample(X_train, y_train)

loan_is_bad
0    42161
1     7808
Name: count, dtype: int64

Proportions:
loan_is_bad
0    0.843743
1    0.156257
Name: proportion, dtype: float64

Imbalance ratio in target: 0.1852


In [52]:
# Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_smote)
y_train_t = torch.LongTensor(y_smote)
X_val_t = torch.FloatTensor(X_val)
y_val_t = torch.LongTensor(y_val)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.LongTensor(np.array(y_test))

In [53]:
# Create DataLoaders
# Checking number of entries in each set
print(f"Training samples: {len(X_train)}, Validation samples: {len(X_val)}, Test samples: {len(X_test)}")
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

batch_size = 64
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size, shuffle=False)

Training samples: 34980, Validation samples: 7493, Test samples: 7496
(34980, 9) (34980,)
(7496, 9) (7496,)


In [57]:
# Define the model
class LoansModel(nn.Module):
    def __init__(self, input_dim, num_classes, dropout_rate=0.2):
        super(LoansModel, self).__init__()

        self.network = nn.Sequential(
            # Layer 1
            nn.Linear(input_dim, input_dim*2),
            nn.BatchNorm1d(input_dim*2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            # Layer 2
            nn.Linear(input_dim*2, input_dim),
            nn.BatchNorm1d(input_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            # Output Layer
            nn.Linear(input_dim, 2)
        )

    def forward(self, x):
        return self.network(x)

In [58]:
# Initialize the model, loss function, and optimizer
model = LoansModel(input_dim=X_train.shape[1], num_classes=2).to(device)

criterion = nn.CrossEntropyLoss()

# AdamW adds weight decay directly to the weight update rule for better regularization
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

# Learning Rate Decay: Reduces LR when validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

In [59]:
epochs = 100
patience = 5
best_val_loss = float('inf')
epochs_no_improve = 0
early_stop = False

train_losses, val_losses = [], []

for epoch in range(epochs):
    if early_stop:
        print(f"Early stopping triggered at epoch {epoch}")
        break

    # --- Training Phase ---
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_train_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    # --- Validation Phase ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

    epoch_val_loss = val_loss / len(val_loader.dataset)
    val_losses.append(epoch_val_loss)

    # Update Learning Rate Scheduler based on validation loss
    scheduler.step(epoch_val_loss)

    # --- Early Stopping Logic ---
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        epochs_no_improve = 0
        # Save the best model
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            early_stop = True

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:03d}/{epochs} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")


Epoch 005/100 | Train Loss: 0.6350 | Val Loss: 0.5498
Epoch 010/100 | Train Loss: 0.6325 | Val Loss: 0.5396
Early stopping triggered at epoch 11


In [60]:
# Evaluation
model.eval()

with torch.no_grad(): # turn off the gradient (stop training)
    y_pred = model(X_test_t) # pass the X_test data

    # if the output < 0.5 then class 0 and else class 1
    y_pred = (y_pred >= 0.5).float()
    accuracy = accuracy_score(y_test_t.numpy(), y_pred.numpy())
    print(f'Accuracy: {round(accuracy, 4)}')

# Convert tensors to NumPy arrays
y_test_np = y_test.numpy()
y_pred_np = y_pred.numpy()

CM.from_predictions(y_test_np, y_pred_np)

predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Create DataFrame
results_df = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
results_df

TypeError: linear(): argument 'input' (position 1) must be Tensor, not numpy.ndarray

In [ ]:
# Load the best model weights found during training
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

correct = 0
total = 0
test_loss = 0.0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        test_loss += loss.item() * inputs.size(0)

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Loss: {test_loss / len(test_loader.dataset):.4f}")
print(f"Test Accuracy: {(100 * correct / total):.2f}%")

# Plotting the learning curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.axvline(x=len(train_losses) - epochs_no_improve - 1, color='r', linestyle='--', label='Best Model')
plt.title('Loss Curves over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()